# Tutorial 5: Causal Gene Exploration

This notebook goes **beyond clustering** to explore the biological meaning behind causal gene scores. You'll learn:

1. How CauST scores each gene
2. How to compare causal vs. non-causal genes
3. How to interpret intervention effects
4. How to export results for downstream analysis (pathway enrichment, etc.)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import pandas as pd

from caust import CauST
from caust.data.loader import load_and_preprocess
from caust.causal.intervention import apply_intervention
from caust.causal.scorer import compute_perturbation_causal_scores, compute_gradient_causal_scores
from caust.visualize.plots import plot_causal_scores, plot_intervention_effect

## 1. Create Synthetic Data with Known Ground-Truth Genes

We embed **3 classes** of genes:
- **Causal** (gene_0 – gene_19): strongly differentiate the spatial domains
- **Noisy** (gene_20 – gene_79): random noise, no spatial signal
- **Confounding** (gene_80 – gene_99): correlated with a hidden batch variable, NOT with domains

In [ ]:
rng = np.random.default_rng(42)
n_obs, n_vars, n_domains = 300, 100, 3
obs_per = n_obs // n_domains

X_parts, coord_parts, labels = [], [], []
for i in range(n_domains):
    base = rng.poisson(5, (obs_per, n_vars)).astype(np.float32)
    # Causal genes: domain i has elevated expression in genes [i*7 : i*7+7]
    causal_start = i * 7
    base[:, causal_start:causal_start + 7] += 12 * (i + 1)
    # Confounding genes: correlated with an unrelated batch variable
    batch_val = rng.normal(i * 3, 1, obs_per)
    base[:, 80:100] += batch_val[:, None]
    X_parts.append(base)
    angle = 2 * np.pi * i / n_domains
    coord_parts.append(rng.normal([10 * np.cos(angle), 10 * np.sin(angle)], 2.0, (obs_per, 2)))
    labels.extend([str(i)] * obs_per)

X = np.vstack(X_parts)
adata = ad.AnnData(X=csr_matrix(X))
adata.obs_names = [f'spot_{i}' for i in range(n_obs)]
adata.var_names = [f'gene_{j}' for j in range(n_vars)]
adata.obsm['spatial'] = np.vstack(coord_parts)
adata.obs['ground_truth'] = labels

# Tag gene categories for later analysis
gene_category = []
for j in range(n_vars):
    if j < 20:
        gene_category.append('causal')
    elif j < 80:
        gene_category.append('noisy')
    else:
        gene_category.append('confounding')
adata.var['category'] = gene_category

adata_pp = load_and_preprocess(adata, n_top_genes=80, min_genes=5, min_cells=1)
print(f'Preprocessed: {adata_pp.n_obs} spots × {adata_pp.n_vars} genes')
print(f'Gene categories kept: {adata_pp.var["category"].value_counts().to_dict()}')

## 2. Run CauST and Get Gene Scores

In [ ]:
model = CauST(
    n_causal_genes=30, n_clusters=3, epochs=80,
    scoring_method='perturbation', alpha=0.5,
    filter_mode='filter_and_reweight', verbose=True,
)
result = model.fit_transform(adata_pp.copy())

# Get the full ranked list
all_scores = model.get_causal_scores()
print(f'\nTotal scored genes: {len(all_scores)}')
print(f'Top 5: {model.get_top_causal_genes(5)}')

## 3. Analyse Score Distribution by Gene Category

The key question: does CauST assign **higher scores to truly causal genes** and **lower scores to noisy/confounding genes**?

In [ ]:
score_df = pd.DataFrame(list(all_scores.items()), columns=['gene', 'score'])
# Map categories
score_df['category'] = score_df['gene'].map(
    adata_pp.var['category'].to_dict() if 'category' in adata_pp.var.columns
    else {g: 'unknown' for g in score_df['gene']}
)
score_df = score_df.sort_values('score', ascending=False).reset_index(drop=True)

# Summary stats
summary = score_df.groupby('category')['score'].agg(['mean', 'median', 'std', 'count'])
print(summary.to_string())
print(f'\nTop 10 genes:')
print(score_df.head(10).to_string(index=False))

In [ ]:
# Box plot of scores by category
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

categories = ['causal', 'noisy', 'confounding']
colors = {'causal': '#2196F3', 'noisy': '#9E9E9E', 'confounding': '#F44336'}

# Box plot
data_by_cat = [score_df[score_df['category'] == c]['score'].values for c in categories if c in score_df['category'].values]
cats_present = [c for c in categories if c in score_df['category'].values]
bp = axes[0].boxplot(data_by_cat, labels=cats_present, patch_artist=True)
for patch, cat in zip(bp['boxes'], cats_present):
    patch.set_facecolor(colors.get(cat, '#999'))
axes[0].set_ylabel('Causal Score')
axes[0].set_title('Score Distribution by Gene Category')

# Ranked bar chart (top 30)
top30 = score_df.head(30)
bar_colors = [colors.get(c, '#999') for c in top30['category']]
axes[1].barh(range(len(top30)), top30['score'].values, color=bar_colors)
axes[1].set_yticks(range(len(top30)))
axes[1].set_yticklabels(top30['gene'].values, fontsize=7)
axes[1].invert_yaxis()
axes[1].set_xlabel('Causal Score')
axes[1].set_title('Top 30 Genes (blue=causal, grey=noisy, red=confounding)')

plt.tight_layout()
plt.show()

In [ ]:
from caust.causal.intervention import compute_global_disruption

# Pick top causal gene and a middle-ranked noisy gene
top_gene = score_df.iloc[0]['gene']
noisy_genes = score_df[score_df['category'] == 'noisy']
mid_noisy = noisy_genes.iloc[len(noisy_genes) // 2]['gene'] if len(noisy_genes) > 0 else score_df.iloc[-1]['gene']

gene_list = list(adata_pp.var_names)
top_idx = gene_list.index(top_gene)
noisy_idx = gene_list.index(mid_noisy)

X_np = adata_pp.X.toarray() if hasattr(adata_pp.X, 'toarray') else np.array(adata_pp.X)

# Zero-out intervention
X_ko_top = apply_intervention(X_np, top_idx, method='zero_out')
X_ko_noisy = apply_intervention(X_np, noisy_idx, method='zero_out')

# Measure disruption
disr_top = compute_global_disruption(X_np, X_ko_top)
disr_noisy = compute_global_disruption(X_np, X_ko_noisy)

print(f'Knockout {top_gene} (causal):  disruption = {disr_top:.4f}')
print(f'Knockout {mid_noisy} (noisy): disruption = {disr_noisy:.4f}')
print(f'Ratio: {disr_top / max(disr_noisy, 1e-8):.2f}x')

## 5. Compare Scoring Methods: Perturbation vs. Gradient

In [ ]:
model_grad = CauST(
    n_causal_genes=30, n_clusters=3, epochs=80,
    scoring_method='gradient', alpha=0.5,
    filter_mode='filter_and_reweight', verbose=False,
)
result_grad = model_grad.fit_transform(adata_pp.copy())

scores_pert = model.get_causal_scores()
scores_grad = model_grad.get_causal_scores()

# Align genes
common = sorted(set(scores_pert.keys()) & set(scores_grad.keys()))
pert_vals = np.array([scores_pert[g] for g in common])
grad_vals = np.array([scores_grad[g] for g in common])

corr = np.corrcoef(pert_vals, grad_vals)[0, 1]

fig, ax = plt.subplots(figsize=(5, 5))
cat_map = adata_pp.var['category'].to_dict() if 'category' in adata_pp.var.columns else {}
for g, pv, gv in zip(common, pert_vals, grad_vals):
    c = colors.get(cat_map.get(g, ''), '#999')
    ax.scatter(pv, gv, c=c, s=20, alpha=0.7)
ax.set_xlabel('Perturbation Score')
ax.set_ylabel('Gradient Score')
ax.set_title(f'Scoring Method Comparison (r={corr:.3f})')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[c], label=c) for c in categories if c in set(cat_map.values())]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

## 6. Filter Mode Comparison

CauST supports 3 filter modes. Let's see how each one affects clustering quality.

In [ ]:
from caust.evaluate.metrics import compute_ari, compute_nmi

filter_modes = ['filter', 'reweight', 'filter_and_reweight']
filter_results = {}
for fm in filter_modes:
    m = CauST(n_causal_genes=30, n_clusters=3, epochs=60,
              scoring_method='gradient', filter_mode=fm, verbose=False)
    res = m.fit_transform(adata_pp.copy())
    pred = res.obs['caust_domain'].astype(int).values
    true = adata_pp.obs.loc[res.obs_names, 'ground_truth'].values
    ari = compute_ari(pred, true)
    nmi = compute_nmi(pred, true)
    filter_results[fm] = {'ari': ari, 'nmi': nmi}
    print(f'{fm:25s}  ARI={ari:.4f}  NMI={nmi:.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(6, 3.5))
x = np.arange(len(filter_modes))
ari_vals = [filter_results[fm]['ari'] for fm in filter_modes]
nmi_vals = [filter_results[fm]['nmi'] for fm in filter_modes]
ax.bar(x - 0.15, ari_vals, 0.3, label='ARI', color='steelblue')
ax.bar(x + 0.15, nmi_vals, 0.3, label='NMI', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(filter_modes, fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Filter Mode Comparison')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 7. Export Scores for Downstream Analysis

Export the ranked gene list as a CSV. It can be used for:
- **Gene Ontology / Pathway Enrichment** (DAVID, Enrichr, GSEA)
- **Comparison with known marker databases** (PanglaoDB, CellMarker)
- **Publication-ready supplementary tables**

In [ ]:
export_df = score_df[['gene', 'score', 'category']].copy()
export_df['rank'] = range(1, len(export_df) + 1)
export_df['selected'] = export_df['gene'].isin([g for g, _ in model.get_top_causal_genes(30)])

import os
os.makedirs('output', exist_ok=True)
export_df.to_csv('output/causal_gene_scores.csv', index=False)

print(f'Exported {len(export_df)} genes to output/causal_gene_scores.csv')
print(export_df.head(10).to_string(index=False))

## Summary

| Analysis | Purpose |
|----------|---------|
| Score by category | Validates CauST identifies true causal genes |
| Intervention effect | Shows mechanistic impact of gene knockout |
| Perturbation vs. gradient | Two complementary scoring approaches |
| Filter mode comparison | `filter_and_reweight` typically best |
| CSV export | Enables downstream biological interpretation |

### Key takeaways
- CauST assigns **higher scores** to genes that genuinely drive spatial domain structure.
- **Confounders** (donor-specific, batch-correlated) are scored low.
- The perturbation and gradient methods are **complementary** — if both rank a gene high, confidence is strong.
- Use the exported CSV for pathway enrichment or comparing against known marker gene databases.

In [ ]:
# Box plot of scores by category
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

categories = ['causal', 'noisy', 'confounding']
colors = {'causal': '#2196F3', 'noisy': '#9E9E9E', 'confounding': '#F44336'}

# Box plot
data_by_cat = [score_df[score_df['category'] == c]['score'].values for c in categories if c in score_df['category'].values]
cats_present = [c for c in categories if c in score_df['category'].values]
bp = axes[0].boxplot(data_by_cat, labels=cats_present, patch_artist=True)
for patch, cat in zip(bp['boxes'], cats_present):
    patch.set_facecolor(colors.get(cat, '#999'))
axes[0].set_ylabel('Causal Score')
axes[0].set_title('Score Distribution by Gene Category')

# Ranked bar chart (top 30)
top30 = score_df.head(30)
bar_colors = [colors.get(c, '#999') for c in top30['category']]
axes[1].barh(range(len(top30)), top30['score'].values, color=bar_colors)
axes[1].set_yticks(range(len(top30)))
axes[1].set_yticklabels(top30['gene'].values, fontsize=7)
axes[1].invert_yaxis()
axes[1].set_xlabel('Causal Score')
axes[1].set_title('Top 30 Genes (blue=causal, grey=noisy, red=confounding)')

plt.tight_layout()
plt.show()